# Final football run: V4-equivalent, 01:00–11:00
Run setup, then preflight. The third code cell alone starts the 15,000-frame GPU job. Do not use Run all. This workflow writes five 3,000-frame chunks to a new timestamped Google Drive folder.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import drive
import json, os, shutil, subprocess, sys
ARCHIVE = Path('/content/sports_vision_v4_final.zip')
ROOT = Path('/content/sports-vision-v4-final')
DRIVE_ROOT = Path('/content/drive/MyDrive/Sports Vision Tracking')
VIDEO = DRIVE_ROOT / 'Data/Football.mp4'
OUTPUTS = DRIVE_ROOT / 'outputs/football'
assert ARCHIVE.is_file(), f'Upload {ARCHIVE.name} to /content first'
assert not ROOT.exists(), f'{ROOT} already exists; use a fresh runtime'
drive.mount('/content/drive')
assert VIDEO.is_file(), f'Missing source video: {VIDEO}'
with ZipFile(ARCHIVE) as bundle:
    required = {'scripts/track_football_final.py', 'configs/football_config.yaml', 'src/common/video_loader.py', 'requirements.txt'}
    assert required.issubset(bundle.namelist()), f'Missing ZIP entries: {required - set(bundle.namelist())}'
    bundle.extractall(ROOT)
link = ROOT / 'Data/Football.mp4'
link.parent.mkdir(parents=True, exist_ok=True)
link.symlink_to(VIDEO)
assert link.resolve(strict=True) == VIDEO.resolve(strict=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements.txt')], check=True)
print('Setup complete; no inference started.')


In [ ]:
# Preflight only: no model loading or inference.
import cv2, torch, yaml
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'Select a Colab T4 runtime'
assert VIDEO.is_file() and VIDEO.stat().st_size > 0
cap = cv2.VideoCapture(str(VIDEO))
assert cap.isOpened(), 'Cannot open Football.mp4'
fps, frames = cap.get(cv2.CAP_PROP_FPS), cap.get(cv2.CAP_PROP_FRAME_COUNT)
cap.release()
assert abs(fps - 25) <= .01 and frames >= 16500, (fps, frames)
cfg = yaml.safe_load((ROOT / 'configs/football_config.yaml').read_text())
assert (cfg['segment']['start_frame'], cfg['segment']['end_frame']) == (1500, 16500)
assert len(cfg['chunks']) == 5 and all(c['expected_frames'] == 3000 for c in cfg['chunks'])
assert OUTPUTS.is_dir() and os.access(OUTPUTS, os.W_OK), f'Output path unavailable: {OUTPUTS}'
free_local_gb = shutil.disk_usage('/content').free / (1024 ** 3)
free_mount_gb = shutil.disk_usage(OUTPUTS).free / (1024 ** 3)
assert free_local_gb >= 2, f'Less than 2 GiB local free space: {free_local_gb:.1f} GiB'
assert free_mount_gb >= 3, f'Less than 3 GiB reported free on Drive mount: {free_mount_gb:.1f} GiB'
print(f'Preflight passed: T4; {frames:.0f} source frames at {fps:.2f} FPS; local {free_local_gb:.1f} GiB, mount {free_mount_gb:.1f} GiB free.')
print('Check your actual Google Drive quota separately; mounted filesystem space may not equal cloud quota.')


## Full 10-minute GPU job — run the next cell only when ready
This call starts YOLOv8s inference with ball fallback and one continuous ByteTrack session. It creates a new timestamped folder. If interrupted, completed chunks remain for inspection, but restart the entire interval in a new folder for globally consistent IDs; no tracker/classifier state checkpoint is saved.


In [ ]:
from datetime import datetime, timezone
RUN = OUTPUTS / ('final_v4_equivalent_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
assert not RUN.exists(), f'Output already exists: {RUN}'
print('Starting full 15,000-frame job:', RUN, flush=True)
subprocess.run([sys.executable, str(ROOT / 'scripts/track_football_final.py'), '--video', str(VIDEO), '--device', 'cuda:0', '--ball-fallback', '--output-dir', str(RUN)], check=True)


In [ ]:
# Read-only artifact check; run only after the job completes.
summary = json.loads((RUN / 'metrics.json').read_text())
assert summary['frames_processed'] == 15000
for i in range(1, 6):
    part = RUN / f'chunk_{i:02d}'
    metrics = json.loads((part / 'metrics.json').read_text())
    assert metrics['frames_processed'] == 3000
    assert metrics['start_frame'] == 1500 + (i - 1) * 3000
    assert metrics['end_frame_exclusive'] == 1500 + i * 3000
    for name in ('annotated.mp4', 'tracks.csv', 'tracks.json', 'pitch_diagnostics.jsonl', 'pitch_diagnostics.mp4'):
        assert (part / name).stat().st_size > 0, part / name
    print(part.name, metrics['start_frame'], metrics['end_frame_exclusive'] - 1, metrics['unique_person_track_ids'])
print('All five chunks present:', RUN)
